In [1]:
import pandas as pd

In [2]:
movies = pd.read_csv("movies.csv")
credits = pd.read_csv("credits.csv")

# Merge on ID
df = movies.merge(credits, on="title")

In [3]:
df = df[['movie_id','title','overview','genres','keywords','cast','crew']]

In [4]:
import ast

def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

df['genres'] = df['genres'].apply(convert)
df['keywords'] = df['keywords'].apply(convert)

In [5]:
def get_cast(text):
    L = []
    for i in ast.literal_eval(text)[:3]:
        L.append(i['name'])
    return L

df['cast'] = df['cast'].apply(get_cast)

In [6]:
def fetch_director(text):
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            return [i['name']]
    return []

df['crew'] = df['crew'].apply(fetch_director)

In [7]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

df['cast'] = df['cast'].apply(collapse)
df['crew'] = df['crew'].apply(collapse)
df['genres'] = df['genres'].apply(collapse)
df['keywords'] = df['keywords'].apply(collapse)

In [8]:
df['overview'] = df['overview'].fillna("")
df['overview'] = df['overview'].apply(lambda x: x.split())

In [9]:
df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['crew']

In [10]:
df['tags'] = df['tags'].apply(lambda x: " ".join(x))
df['tags'] = df['tags'].apply(lambda x: x.lower())

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

In [13]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [15]:
def recommend(movie):
    movie_index = df[df['title'] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)),
                         reverse=True,
                         key=lambda x: x[1])[1:6]

    for i in movies_list:
        print(df.iloc[i[0]].title)

In [16]:
recommend("Avatar")

Titan A.E.
Small Soldiers
Ender's Game
Independence Day
Aliens vs Predator: Requiem


In [19]:
import pickle

# Save movies dataframe
pickle.dump(df, open("movies.pkl", "wb"))

# Save similarity matrix
pickle.dump(similarity, open("similarity.pkl", "wb"))